# The crisis of the bubble's chaotic attractor at $\varepsilon_d = 10\,g_2$: what does it collide with? (23/10/2026)

Follow-up of `satellite_start_bubble_edge.ipynb` (section 4) and `Hill_period2_branch_II.ipynb`. At $\varepsilon_d = 10\,g_2$, $\kappa_b = \pi/10.4$ (code), the bubble's chaotic attractor ($\lambda_1 = 5.6\,\kappa_b$ at 100 MHz)
stops being an attractor between 101.2 MHz (no escape of 80 runs within $1490/\kappa_b$) and 101.5 MHz (15 of 80 escape); a power-law fit of the lifetimes gave $g_c \simeq 101.4$ MHz, $\gamma \simeq 1.3$.
Every escape lands on the **$\omega_a/2$ orbit** (node side of the saddle-node at 99.970 MHz), which is stable on 99.97–100.24 and **101.20**–106.71 MHz.

Two scenarios are tested:
- **(A) Boundary crisis by collision with a saddle.** The attractor touches the stable manifold of a saddle orbit on the boundary between its basin and the node's, the natural candidate being the saddle side of the $\omega_a/2$ fold.
  Then, below $g_c$, one branch of the saddle's unstable manifold goes to the node and the other to the chaotic attractor. The attractor's distance to the saddle goes to zero at $g_c$, and the escaping trajectories pass close to the saddle.
- **(B) A hole opening inside the attractor.** The node becomes stable inside the region visited by the attractor, at the Neimark–Sacker point $g_\mathrm{NS} = 101.20$ MHz. If that Neimark–Sacker is subcritical, its basin is bounded by an unstable invariant torus whose size grows from zero at $g_\mathrm{NS}$.
  Then the chaos becomes transient at $g_\mathrm{NS}$ itself. The escape rate is the attractor's natural measure inside the hole, and the escaping trajectories enter the node's basin directly, without passing near the saddle.

Plan:
1. Candidate orbits from harmonic balance (Python) on a grid of $g_2$: $\omega_a/2$ node and saddle, branch I, satellite; section states, multipliers, eigenvectors. Precise Neimark–Sacker points of the node.
2. Time integrations (`crisis_collision.jl`, Julia, ≈ 10 min on 8 threads; skipped if the output exists and `RUN_JULIA = False`):
   - near the node around $g_\mathrm{NS}$ (type of the Neimark–Sacker, size of the node's basin);
   - the unstable manifold of the $\omega_a/2$ saddle;
   - long runs on the attractor below $g_\mathrm{NS}$ (distances to every candidate);
   - escape routes above $g_\mathrm{NS}$.
3. The lifetimes of `bubble_crisis_lifetime_22_10_2026.json` and of this run, against both scenarios.

**Conventions.** As in `Hill_period2_branch_II.ipynb`. Section $\tau = 0 \bmod 2\pi$, state $y = [x_a, x_b, dx_a/d\tau, dx_b/d\tau]$ ($x^\lambda$ units), distances are Euclidean in $y$.
Each period-2 orbit is represented by its two section points and those of its partner (shift by $T_a/2$, i.e. the orbit at $\tau = \pi, 3\pi$). Times are in memory periods $T_a$ or in $1/\kappa_b$ (code convention; $1/\kappa_b = 13.4\,T_a$).

In [ ]:
import json, time, warnings
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
from scipy.optimize import root
from scipy.integrate import solve_ivp
import sys
sys.path.insert(0, "/Users/tmalas/Desktop/Dev/Simulations_local/Code_python")
import Plot_setting
from Plot_setting import *
warnings.filterwarnings("ignore", category=RuntimeWarning)

OUT_DIR = "/Users/tmalas/Desktop/Dev/Simulations_local/classical_simulations/data/"
FIG_DIR = "/Users/tmalas/Desktop/Dev/Simulations_local/classical_simulations/figures/torus_crisis/"

w_a = 25.338776456203686
w_b = 2*w_a
phi_a, phi_b = 0.11, 0.204
E_J = 37.12*2*np.pi*1.5
kappa_b = np.pi/10.4                 # code convention
G0 = E_J*phi_a**2*phi_b              # g2 = s G0, s = sin(eps_p)
R = 10.0                             # eps_d / g2 (row of bubble_crisis_lifetime.jl)
n = 2
M = np.diag([1.0, 0.25])
K = np.eye(2)
N_H, N_H2 = 16, 32                   # harmonics: period 1, period 2 (N_H2 = 2 N_H, so the satellite embeds exactly)
G_C = 101.4                          # crisis of the bubble's attractor (satellite_start_bubble_edge, section 4)

def MHz(g):
    return g/2/np.pi*1e3
g2_max = MHz(G0)

## Harmonic balance and continuation
Copied from `Hill_period2_branch_II.ipynb`.

In [ ]:
class HB:
    def __init__(self, N_H, kfac=1.0, N=None):
        self.N_H, self.N = N_H, N or 64*N_H
        self.C = np.diag([0.0, kfac*kappa_b/w_b])
        self.L = self.build_L(M, self.C)
        t = np.linspace(0, 2*np.pi, self.N, endpoint=False)
        k = np.arange(1, N_H + 1)
        Phi = np.empty((self.N, 2*N_H + 1))
        Phi[:, 0] = 1/np.sqrt(2)
        Phi[:, 1::2] = np.sin(np.outer(t, k))
        Phi[:, 2::2] = np.cos(np.outer(t, k))
        self.Gamma = np.kron(Phi, np.eye(n))
        self.Gamma_pinv = (2/self.N)*self.Gamma.T
        self.drive_harmonic = 2

    def build_L(self, M_, C_):
        L = np.zeros((n*(2*self.N_H + 1),)*2)
        L[:n, :n] = K
        for k in range(1, self.N_H + 1):
            A, Bk = K - k**2*M_, k*C_
            s_, c_ = 2*k - 1, 2*k
            L[s_*n:(s_+1)*n, s_*n:(s_+1)*n] = A
            L[s_*n:(s_+1)*n, c_*n:(c_+1)*n] = -Bk
            L[c_*n:(c_+1)*n, s_*n:(s_+1)*n] = Bk
            L[c_*n:(c_+1)*n, c_*n:(c_+1)*n] = A
        return L

    def set_s(self, s, r=R):
        self.B = 2*E_J*s*np.array([phi_a**2/w_a, phi_b**2/w_b])          # (beta_a', beta_b')
        self.b_ext = np.zeros(n*(2*self.N_H + 1))
        self.b_ext[2*self.drive_harmonic*n + 1] = -np.sqrt(2)*r*s*G0*phi_b/w_b

    def X(self, z):
        return (self.Gamma @ z).reshape(self.N, n).T

    def residual(self, z):
        X = self.X(z)
        Fnl = -np.sqrt(2)*self.B[:, None]*(np.cos(np.sqrt(2)*(X[0] + X[1])) - 1)[None, :]
        return self.L @ z + self.Gamma_pinv @ Fnl.T.reshape(-1) - self.b_ext

    def jacobian(self, z):
        X = self.X(z)
        sn = np.sin(np.sqrt(2)*(X[0] + X[1]))
        JG = np.einsum('j,ab,jbm->jam', 2*sn, np.outer(self.B, [1, 1]),
                       self.Gamma.reshape(self.N, n, -1)).reshape(n*self.N, -1)
        return self.L + self.Gamma_pinv @ JG

    def solve(self, z0, tol=1e-12):
        sol = root(self.residual, z0, jac=self.jacobian, method="hybr", tol=tol, options={"maxfev": 100*(len(z0) + 1)})
        if not sol.success:
            raise RuntimeError(sol.message)
        return sol.x


class HB2(HB):
    # period-2 orbit: basis over tau_d = tau/2
    def __init__(self, N_H, kfac=1.0, N=None):
        super().__init__(N_H, kfac, N)
        self.L = self.build_L(M/4, self.C/2)
        self.drive_harmonic = 4


def orbit(z, N_H, t):
    # x and dx/dt of both modes (t in the basis time of the HB object)
    k = np.arange(1, N_H + 1)
    out = []
    for zz in (z[::2], z[1::2]):
        s_, c_ = zz[1::2], zz[2::2]
        x = zz[0]/np.sqrt(2) + np.sin(np.outer(t, k)) @ s_ + np.cos(np.outer(t, k)) @ c_
        v = np.cos(np.outer(t, k)) @ (k*s_) - np.sin(np.outer(t, k)) @ (k*c_)
        out.append((x, v))
    return out

def strobe(z, N_H, tau=0.0):
    # period-1 orbit: state [x_a, x_b, v_a, v_b] at time tau
    (x_a, v_a), (x_b, v_b) = orbit(z, N_H, np.array([tau]))
    return np.array([x_a[0], x_b[0], v_a[0], v_b[0]])

def rotating_frame(z, N_H, tau, period=1):
    # a_R = i a e^{i tau}, b_R = b e^{2 i tau} (limit_cycles.ipynb); tau in units of omega_a t
    (x_a, v_a), (x_b, v_b) = orbit(z, N_H, tau/period)
    v_a, v_b = v_a/period, v_b/period                                   # d/dtau
    return 1j*(x_a + 1j*v_a)/(np.sqrt(2)*phi_a)*np.exp(1j*tau), (x_b + 1j*v_b/2)/(np.sqrt(2)*phi_b)*np.exp(2j*tau)

def odd_content(z, N_H):
    # odd harmonics of the memory in the period-2 basis: zero on a period-1 orbit
    za = z[::2]
    return np.sqrt(sum(za[2*k - 1]**2 + za[2*k]**2 for k in range(1, N_H + 1, 2)))

def monodromy(hb, z, period=1):
    Minv, base = np.linalg.inv(M), np.outer(hb.B, [1, 1])
    def rhs(t, Y):
        (x_a, _), (x_b, _) = orbit(z, hb.N_H, np.atleast_1d(t)/period)
        A = np.zeros((4, 4)); A[:2, 2:] = np.eye(2)
        A[2:, :2] = -Minv @ (K + 2*np.sin(np.sqrt(2)*(x_a[0] + x_b[0]))*base); A[2:, 2:] = -Minv @ hb.C                       # hb.C is in tau units for both bases
        return (A @ Y.reshape(4, 4)).reshape(-1)
    sol = solve_ivp(rhs, (0, 2*np.pi*period), np.eye(4).reshape(-1), method="DOP853", rtol=1e-10, atol=1e-12)
    return np.linalg.eigvals(sol.y[:, -1].reshape(4, 4))

def instability(mu):
    # type of the leading multiplier
    m = mu[np.argmax(np.abs(mu))]
    if abs(m) < 1:
        return "stable"
    if abs(m.imag) > 1e-6:
        return "NS"
    return "PD" if m.real < 0 else "saddle"

def full_rhs(s, kfac=1.0, r=R):
    beta = 2*E_J*s*np.array([phi_a**2/w_a, phi_b**2/w_b])
    eps_d_p, kp = r*s*G0*phi_b/w_b, kfac*kappa_b/w_b
    def f(t, Y):
        x_a, x_b, v_a, v_b = Y
        g = np.cos(np.sqrt(2)*(x_a + x_b)) - 1
        return [v_a, v_b, -x_a + np.sqrt(2)*beta[0]*g, 4*(-x_b - np.sqrt(2)*eps_d_p*np.cos(2*t) + np.sqrt(2)*beta[1]*g - kp*v_b)]
    return f


# analytic satellite (App. "Satellites", quadratic order), only used as the first seed
yb_grid = np.linspace(1e-3, 3.0, 3000)
def satellite_hb(s, r=R):
    kp = kappa_b/w_b
    ba, bb = 2*E_J*s*phi_a**2/w_a, 2*E_J*s*phi_b**2/w_b
    ed, bS = r*s*G0*phi_b/w_b, ba + bb
    gam = 1 + bb/(ba*(1 + kp**2))
    eeff = lambda y: np.sqrt(np.maximum(ed**2 - 4*kp**2*y**2, 0.0))
    def y_of(yb):
        y = yb + ba*ed/(2*bb)
        for _ in range(30):
            y = yb + ba*eeff(y)/(2*bb)
        return y
    def u_of(yb):
        eta = 2*bb*yb
        return (np.sqrt((1 + eta)**2 + 4*eta**2) - (1 + eta))/(2*eta)
    def x2_of(yb):
        u = u_of(yb)
        return (2*(1 + u)*yb**2 - eeff(y_of(yb))/bb)/(1 + 2*u)
    def resid(yb):
        u = u_of(yb)
        return yb*(1 + u) - 2*bS*((1 + (1 + u)**2/2)*yb**2 + (1 + u**2)*x2_of(yb))
    rv = resid(yb_grid)
    for k in np.where(rv[:-1]*rv[1:] <= 0)[0]:
        lo, hi = yb_grid[k], yb_grid[k + 1]
        for _ in range(60):
            mid = (lo + hi)/2
            if resid(lo)*resid(mid) <= 0:
                hi = mid
            else:
                lo = mid
        yb = (lo + hi)/2
        x2 = x2_of(yb)
        if x2 <= 0:
            continue
        u, y, x = u_of(yb), y_of(yb), np.sqrt(x2)
        if 2*kp*abs(y) > ed:
            continue
        th = -np.arcsin(2*kp*y/ed)/2
        a1 = -(1 + u)*yb/(2*gam)
        return dict(a0=x*np.exp(1j*th), a1=a1, am1=-ba*eeff(y)/(2*bb)*np.exp(2j*th),
                    b0=y*np.exp(2j*th), bm1=u*x*np.exp(3j*th), b2=bb/ba*a1)

def seed_z(S, N_H):
    za, zb = np.zeros(2*N_H + 1), np.zeros(2*N_H + 1)
    za[0] = 2*np.real(S["a1"])
    za[1], za[2] = np.sqrt(2)*np.imag(S["a0"]), np.sqrt(2)*np.real(S["a0"])
    za[3], za[4] = np.sqrt(2)*np.imag(S["am1"]), np.sqrt(2)*np.real(S["am1"])
    zb[0] = 2*np.real(S["b2"])
    zb[3], zb[4] = np.sqrt(2)*np.imag(S["b0"]), np.sqrt(2)*np.real(S["b0"])
    zb[5], zb[6] = np.sqrt(2)*np.imag(S["bm1"]), np.sqrt(2)*np.real(S["bm1"])
    z = np.zeros(n*(2*N_H + 1)); z[::2], z[1::2] = za, zb
    return z

In [ ]:
def newton(hb, z, tol=1e-11, maxit=30):
    for _ in range(maxit):
        r_ = hb.residual(z)
        if np.linalg.norm(r_) < tol:
            return z
        z = z - np.linalg.solve(hb.jacobian(z), r_)
    if np.linalg.norm(hb.residual(z)) < 1e3*tol:
        return z
    raise RuntimeError("Newton did not converge")

def embed(z1, N1, N2):
    # period-1 coefficients in the period-2 basis: harmonic j of tau -> harmonic 2j of tau_d
    z2 = np.zeros(n*(2*N2 + 1)); z2[:n] = z1[:n]
    for j in range(1, min(N1, N2//2) + 1):
        z2[(4*j - 1)*n:4*j*n] = z1[(2*j - 1)*n:2*j*n]
        z2[4*j*n:(4*j + 1)*n] = z1[2*j*n:(2*j + 1)*n]
    return z2

ODD = np.zeros(n*(2*N_H2 + 1), bool)                    # coefficients of the odd harmonics (both modes)
for k in range(1, N_H2 + 1, 2):
    ODD[(2*k - 1)*n:(2*k + 1)*n] = True

PS = 100.0                                              # continuation parameter p = g2 / (100 MHz)
def extended(hb2):
    def setp(p): hb2.set_s(p*PS/g2_max)
    def F(y):
        setp(y[-1]); return hb2.residual(y[:-1])
    def dF(y, h=1e-7):
        setp(y[-1]); Jz, r_ = hb2.jacobian(y[:-1]), hb2.residual(y[:-1])
        setp(y[-1] + h)
        return np.column_stack([Jz, (hb2.residual(y[:-1]) - r_)/h])
    return F, dF

def arclength(hb2, y, t, ds0=2e-3, ds_max=3e-3, g_lo=60.0, g_hi=140.0, n_max=5000, stop_odd=True):
    # pseudo-arclength from y = (z, p) along t; stops when g2 leaves [g_lo, g_hi] or (stop_odd) the odd harmonics vanish again
    F, dF = extended(hb2)
    def corrector(y0, t, ds):
        w = y0 + ds*t
        for _ in range(20):
            G = np.append(F(w), t @ (w - y0) - ds)
            if np.linalg.norm(G) < 1e-11:
                return w
            w = w - np.linalg.solve(np.vstack([dF(w), t]), G)
        if np.linalg.norm(np.append(F(w), t @ (w - y0) - ds)) < 1e-9:
            return w
        raise RuntimeError
    ds, hist, end = ds0, [y.copy()], "n_max reached"
    for it in range(n_max):
        try:
            yn = corrector(y, t, ds)
        except (RuntimeError, np.linalg.LinAlgError):
            ds /= 2
            if ds < 1e-8:
                end = f"step too small at {y[-1]*PS:.2f} MHz"; break
            continue
        tn = np.linalg.svd(dF(yn))[2][-1]
        t = -tn if tn @ t < 0 else tn
        y = yn; hist.append(y.copy()); ds = min(1.3*ds, ds_max)
        g = y[-1]*PS
        if not g_lo < g < g_hi:
            end = f"leaves [{g_lo:g}, {g_hi:g}] MHz"; break
        if stop_odd and it > 20 and odd_content(y[:-1], hb2.N_H) < 1e-5:
            end = f"odd harmonics vanish at {g:.3f} MHz"; break
    return np.array(hist), end

def fourier_seed(y_state, s, N_H2, N=2048):
    # one period 2 T_a from a state on the section, projected on the period-2 basis (tau_d = tau/2)
    tt = np.linspace(0, 4*np.pi, N, endpoint=False)
    sol = solve_ivp(full_rhs(s), (0, 4*np.pi), y_state, t_eval=tt, method="DOP853", rtol=1e-11, atol=1e-12)
    td, k = tt/2, np.arange(1, N_H2 + 1)
    z = np.zeros(n*(2*N_H2 + 1))
    for m in range(2):
        zz = np.empty(2*N_H2 + 1)
        zz[0] = np.sqrt(2)*sol.y[m].mean()
        zz[1::2] = (2/N)*np.sin(np.outer(td, k)).T @ sol.y[m]
        zz[2::2] = (2/N)*np.cos(np.outer(td, k)).T @ sol.y[m]
        z[m::2] = zz
    return z

def state(z, N, tau, period):
    # [x_a, x_b, dx_a/dtau, dx_b/dtau] at tau (x^lambda units of the harmonic balance)
    (x_a, v_a), (x_b, v_b) = orbit(z, N, np.atleast_1d(tau/period))
    return np.array([x_a[0], x_b[0], v_a[0]/period, v_b[0]/period])

def section(z, N, period=2):
    # a on the section tau = 0 mod 2 pi (amp of bubble_crisis_lifetime.jl), and for the partner orbit (tau = pi mod 2 pi)
    taus = 2*np.pi*np.arange(period)
    amp = lambda Y: (Y[0] + 1j*Y[2])/(np.sqrt(2)*phi_a)
    return np.array([amp(state(z, N, t, period)) for t in taus]), np.array([amp(state(z, N, t + np.pi, period)) for t in taus])

def describe(H, N, period=2):
    # multipliers, type, odd content, section, <|a_R|^2> at every point of a branch H (rows (z, p))
    hb_ = HB2(N) if period == 2 else HB(N)
    tt = np.linspace(0, 2*np.pi*period, 300*period, endpoint=False)
    out = dict(g2=H[:, -1]*PS, mu=[], kind=[], odd=[], sec=[], sec_partner=[], a2=[])
    for y in H:
        hb_.set_s(y[-1]*PS/g2_max)
        mu = monodromy(hb_, y[:-1], period); mu = mu[np.argsort(-np.abs(mu))]
        s0, s1 = section(y[:-1], N, period)
        out["mu"].append(mu); out["kind"].append(instability(mu)); out["odd"].append(odd_content(y[:-1], N) if period == 2 else 0.0)
        out["sec"].append(s0); out["sec_partner"].append(s1)
        out["a2"].append(np.mean(np.abs(rotating_frame(y[:-1], N, tt, period)[0])**2))
    return {k: np.array(v) for k, v in out.items()}

def unstable_set(mu):
    # multipliers outside the unit circle: "+" real > 1, "-" real < -1, "c" complex pair (counted once)
    out = [("c" if abs(m.imag) > 1e-6 else ("+" if m.real > 0 else "-")) for m in mu if abs(m) > 1 and m.imag >= -1e-6]
    return "".join(sorted(out)) or "stable"

def transitions(D):
    # changes of the set of unstable multipliers along a branch; g2 of the crossing interpolated on the multiplier that crosses |mu| = 1
    res, sets = [], [unstable_set(m) for m in D["mu"]]
    for i in range(1, len(sets)):
        if sets[i] != sets[i - 1]:
            a, b = np.sort(np.abs(D["mu"][i - 1])), np.sort(np.abs(D["mu"][i]))
            k = np.argmax(np.abs((a > 1).astype(int) - (b > 1).astype(int))) if np.any((a > 1) != (b > 1)) else None
            g = D["g2"][i - 1] + (1 - a[k])*(D["g2"][i] - D["g2"][i - 1])/(b[k] - a[k]) if k is not None else D["g2"][i]
            res.append((g, sets[i - 1], sets[i]))
    return res

# 1. Candidate orbits on the $g_2$ grid
The period-2 orbits are followed from the seeds stored in `period2_branches_eps_d10_23_10_2026.json`, by natural continuation in steps of 0.02 MHz, downwards from 103 MHz. Each point is checked against the stored odd content, because the saddle and node sides of the fold must not be confused.
For each orbit and $g_2$: the section states (orbit and partner), the multipliers of the map over the period, and the leading eigenvector. The two Neimark–Sacker points of the node are bisected on $\max|\mu| = 1$.

In [ ]:
B2 = json.load(open(OUT_DIR + "torus_crisis/period2_branches_eps_d10_23_10_2026.json"))
G = [100.0, 100.1, 100.2, 100.3, 100.6, 100.9, 101.0, 101.1, 101.15, 101.18, 101.2, 101.22, 101.25, 101.3, 101.35, 101.4, 101.5, 101.8, 102.1, 102.4, 103.0]

def follow(name, gs, dg=0.02):
    # natural continuation along a stored branch through gs (in the given order), checked against the stored odd content
    D = B2[name]; gz = np.array(D["g2_MHz"])[::D["z_every"]]; zz = np.array(D["z"])
    g_all, odd_all = np.array(D["g2_MHz"]), np.array(D["odd"])
    j = np.argmin(np.abs(gz - gs[0])); z, g = zz[j], gz[j]
    hb2 = HB2(N_H2); out = {}
    for gt in gs:
        for gg in np.linspace(g, gt, max(1, int(np.ceil(abs(gt - g)/dg))) + 1)[1:]:
            hb2.set_s(gg/g2_max); z = newton(hb2, z)
        g = gt
        assert abs(odd_content(z, N_H2) - odd_all[np.argmin(np.abs(g_all - g))]) < 0.01, f"{name} jumped branch at {g} MHz"
        out[gt] = z.copy()
    return out

Z = {"node": follow("omega_half_node", G[::-1]), "saddle": follow("omega_half_saddle", G[::-1]), "branch_I": follow("branch_I", G[::-1])}
hb = HB(N_H)
hb.set_s(60/g2_max); z = hb.solve(seed_z(satellite_hb(60/g2_max), N_H))
for g in np.arange(60.5, 100.01, 0.5):
    hb.set_s(g/g2_max); z = newton(hb, z)
Z["satellite"], g_prev = {}, 100.0
for g in G:
    for gg in np.linspace(g_prev, g, 5)[1:]:
        hb.set_s(gg/g2_max); z = newton(hb, z)
    g_prev = g; Z["satellite"][g] = z.copy()

def monodromy_matrix(hb_, z, period):
    Minv, base = np.linalg.inv(M), np.outer(hb_.B, [1, 1])
    def rhs(t, Y):
        (x_a, _), (x_b, _) = orbit(z, hb_.N_H, np.atleast_1d(t)/period)
        A = np.zeros((4, 4)); A[:2, 2:] = np.eye(2)
        A[2:, :2] = -Minv @ (K + 2*np.sin(np.sqrt(2)*(x_a[0] + x_b[0]))*base); A[2:, 2:] = -Minv @ hb_.C
        return (A @ Y.reshape(4, 4)).reshape(-1)
    sol = solve_ivp(rhs, (0, 2*np.pi*period), np.eye(4).reshape(-1), method="DOP853", rtol=1e-11, atol=1e-13)
    return sol.y[:, -1].reshape(4, 4)

ORB = {}
for k, ZZ in Z.items():
    per = 1 if k == "satellite" else 2
    N = N_H if per == 1 else N_H2
    rows = []
    for g in G:
        hb_ = HB(N) if per == 1 else HB2(N); hb_.set_s(g/g2_max)
        w, V = np.linalg.eig(monodromy_matrix(hb_, ZZ[g], per))
        o = np.argsort(-np.abs(w)); w, V = w[o], V[:, o]
        taus = 2*np.pi*np.arange(per)
        rows.append(dict(states=[state(ZZ[g], N, t, per).tolist() for t in taus], partner=[state(ZZ[g], N, t + np.pi, per).tolist() for t in taus],
                         mu_re=w.real.tolist(), mu_im=w.imag.tolist(), v_lead=V[:, 0].real.tolist()))
    ORB[k] = rows
    print(f"{k:10s} max|mu| on the grid: " + " ".join(f"{abs(complex(r['mu_re'][0], r['mu_im'][0])):.3g}" for r in rows))

In [ ]:
def node_maxmu(g, z):
    hb2 = HB2(N_H2); hb2.set_s(g/g2_max); z = newton(hb2, z)
    mu = monodromy(hb2, z, 2); return np.abs(mu).max() - 1, z, mu[np.argmax(np.abs(mu))]

def bisect_ns(lo, hi, z):
    f_lo, z, _ = node_maxmu(lo, z)
    for _ in range(30):
        mid = (lo + hi)/2; f, zm, _ = node_maxmu(mid, z)
        if (f > 0) == (f_lo > 0):
            lo, f_lo, z = mid, f, zm
        else:
            hi = mid
    return (lo + hi)/2, node_maxmu((lo + hi)/2, z)[2]

g_NS1, mu_NS1 = bisect_ns(100.2, 100.3, Z["node"][100.2])
g_NS, mu_NS = bisect_ns(101.1, 101.3, Z["node"][101.1])
print(f"node of the omega_a/2 orbit: Neimark-Sacker at {g_NS1:.4f} MHz (loses stability, arg mu/2pi = {abs(np.angle(mu_NS1))/2/np.pi:.4f}) "
      f"and at g_NS = {g_NS:.4f} MHz (restabilises, arg mu/2pi = {abs(np.angle(mu_NS))/2/np.pi:.4f}); the saddle-node is at {B2['g2_omega_half_fold_MHz']:.4f} MHz")
i = G.index(101.4)
mu_s = np.array(ORB["saddle"][i]["mu_re"]) + 1j*np.array(ORB["saddle"][i]["mu_im"])
print(f"omega_a/2 saddle at 101.4 MHz: multipliers {np.round(mu_s, 4)}")

# 2. Time integrations
Tasks of `crisis_collision.jl` (all on the section; "landed" = within $10^{-3}$ of the node or its partner):
- **ns**: below $g_\mathrm{NS}$, the node kicked by $10^{-5}$ in random directions (distance to the node vs time). Above $g_\mathrm{NS}$, kicks of size $\epsilon$ in 8 random directions: does the trajectory come back to the node without leaving a 0.3-neighbourhood?
- **manifold**: the saddle state $\pm\delta\,v_u$ ($\delta = 10^{-7}, 10^{-6}, 10^{-5}$), up to $10^5\,T_a$: landing time or not;
- **attractor**: 8 runs of $10^5\,T_a$ per $g_2$ from points of the attractor at 100 MHz: histogram of the distance to each candidate, landing time if any;
- **escape**: 24 runs per $g_2$ above $g_\mathrm{NS}$ (up to $2\times10^5\,T_a$), keeping the last 600 iterates before landing.

In [ ]:
import os, subprocess
HERE = "/Users/tmalas/Desktop/Dev/Simulations_local/classical_simulations/studies/torus_crisis/"
TASKS = {"attractor": {"g2_MHz": [100.0, 100.1, 100.2, 100.3, 100.6, 100.9, 101.0, 101.1, 101.15, 101.18, 101.2], "n_traj": 8, "N_Ta": 100000},
         "escape": {"g2_MHz": [101.22, 101.25, 101.3, 101.35, 101.4, 101.5, 101.8, 102.4, 103.0], "n_traj": 24, "N_max_Ta": 200000},
         "manifold": {"g2_MHz": G, "delta": [1e-7, 1e-6, 1e-5], "N_max_Ta": 100000},
         "ns": {"g2_below": [101.1, 101.15, 101.18], "n_below": 4, "N_below_Ta": 50000,
                "g2_above": [101.22, 101.25, 101.3, 101.35, 101.4, 101.5, 101.8, 102.4, 103.0], "eps": np.logspace(-3.5, -0.5, 13).tolist(), "n_kick": 8, "N_kick_Ta": 20000}}
IN_J = OUT_DIR + "torus_crisis/crisis_collision_input_23_10_2026.json"
OUT_J = OUT_DIR + "torus_crisis/crisis_collision_23_10_2026.json"
json.dump({"R": R, "g2_MHz": G, "orbits": ORB, "tasks": TASKS}, open(IN_J, "w"))

RUN_JULIA = False
if RUN_JULIA or not os.path.exists(OUT_J):
    t0 = time.time()
    run = subprocess.run(["julia", "-t", "8", "--project=/Users/tmalas/Desktop/Dev/Simulations_local/Code_Julia", HERE + "crisis_collision.jl", IN_J, OUT_J],
                         capture_output=True, text=True)
    print(run.stdout[-1500:], run.stderr[-1500:], f"({time.time() - t0:.0f} s)")
CC = json.load(open(OUT_J))
RUNS = CC["runs"]; EDG = np.array(CC["edges"]); KT_A = CC["T_a_over_kappa_inv"]
by = lambda task, **kw: [x for x in RUNS if x["task"] == task and all(abs(x[k] - v) < 1e-9 for k, v in kw.items())]
print({t: len(by(t)) for t in ("ns_below", "ns_kick", "manifold", "attractor", "escape")})

# 3. The Neimark–Sacker point of the node and the size of its basin
**Below $g_\mathrm{NS}$** (node unstable): distance to the node after a $10^{-5}$ kick. A supercritical Neimark–Sacker leaves a small stable torus around the node, a subcritical one does not.
**Above $g_\mathrm{NS}$** (node stable): fraction of 8 random kicks of size $\epsilon$ that come back to the node without leaving its 0.3-neighbourhood; $\rho_{50}(g_2)$ is the kick size at which half come back (log interpolation).

In [ ]:
for g in TASKS["ns"]["g2_below"]:
    for x in by("ns_below", g2_MHz=g):
        d = np.array(x["dist_every_10Ta"])
        fate = "left (distance > 1)" if len(d)*10 < TASKS["ns"]["N_below_Ta"] else f"stays within {d.max():.4f} for {TASKS['ns']['N_below_Ta']} T_a"
        print(f"g2 = {g:7.2f} MHz, kick {x['seed']}: {fate}")
EPS = np.array(TASKS["ns"]["eps"])
FR, RHO = {}, {}
for g in TASKS["ns"]["g2_above"]:
    FR[g] = np.array([np.mean([x["returned_Ta"] is not None for x in by("ns_kick", g2_MHz=g, eps=e)]) for e in EPS])
    i = np.where(FR[g] < 0.5)[0][0]
    RHO[g] = np.exp(np.interp(0.5, [FR[g][i], FR[g][i - 1]], [np.log(EPS[i]), np.log(EPS[i - 1])]))
g_rho = np.array(list(RHO)); rho = np.array(list(RHO.values()))
p_rho = np.polyfit(np.log(g_rho - g_NS), np.log(rho), 1)
print("rho_50 (x^lambda units): " + ", ".join(f"{g:g}: {r:.4f}" for g, r in RHO.items()) + f";  rho_50 ~ (g2 - g_NS)^{p_rho[0]:.2f}")

# 4. The unstable manifold of the $\omega_a/2$ saddle
Landing time on the node (in $T_a$) of the saddle state displaced by $\pm\delta\,v_u$; "—" = no landing within $10^5\,T_a$ (below $g_\mathrm{NS}$ and above 100.24 MHz the node is unstable, so nothing can land).
A saddle on the boundary between the node's basin and the attractor's has one branch going straight to the node (a few hundred $T_a$ at most) and the other into the chaotic set.

In [ ]:
G_node = B2["omega_half_node"]
print("  g2 (MHz) | node | " + " | ".join(f"{s}{d:.0e}" for s in "-+" for d in TASKS["manifold"]["delta"]))
MAN = {}
for g in G:
    rr = {(x["sign"], x["delta"]): x["landed_Ta"] for x in by("manifold", g2_MHz=g)}
    MAN[g] = rr
    stable = abs(complex(ORB["node"][G.index(g)]["mu_re"][0], ORB["node"][G.index(g)]["mu_im"][0])) < 1
    print(f"  {g:7.2f} | {'stable' if stable else 'NS    '} | " + " | ".join(f"{rr[(s, d)] if rr[(s, d)] is not None else '—':>6}" for s in (-1, 1) for d in TASKS["manifold"]["delta"]))

# 5. The attractor below $g_\mathrm{NS}$
For each $g_2$, 8 runs of $10^5\,T_a$: landings on the node, the fraction of iterates within 0.1 of the node (a capture by a torus around the node shows up there), the smallest distance to each candidate,
and the attractor's measure $\mu_\mathrm{att}(<r)$ within $r$ of the node (orbit or partner).

In [ ]:
CAND = ("node", "saddle", "branch_I", "satellite")
def cdf_of(rr, k):
    h = np.sum([x["hist"][k] for x in rr], axis=0); return np.cumsum(h)/h.sum()
mu_at = lambda cdf, r: np.interp(np.log(r), np.log(EDG[1:]), cdf)
print(" g2 (MHz) | landed (T_a)                          | runs with > 10 % of iterates within 0.1 of the node | min distance: " + ", ".join(CAND))
for g in TASKS["attractor"]["g2_MHz"]:
    rr = by("attractor", g2_MHz=g)
    land = sorted(x["landed_Ta"] for x in rr if x["landed_Ta"] is not None)
    near = sum(np.cumsum(x["hist"]["node"])[np.searchsorted(EDG[1:], 0.1)]/np.sum(x["hist"]["node"]) > 0.1 for x in rr)
    print(f"  {g:7.2f} | {len(land)}/8 {str(land):32s} | {near}/8 | " + ", ".join(f"{min(x['dmin'][k] for x in rr):.4f}" for k in CAND))
CDF_NS = cdf_of(by("attractor", g2_MHz=101.18) + by("attractor", g2_MHz=101.2), "node")
r_grid = np.array([0.01, 0.02, 0.05, 0.1])
D_loc = np.polyfit(np.log(r_grid), np.log([mu_at(CDF_NS, r) for r in r_grid]), 1)[0]
print("attractor at 101.18-101.2 MHz: mu_att(<r) = " + ", ".join(f"{mu_at(CDF_NS, r):.1e} (r = {r:g})" for r in r_grid) + f";  mu_att ~ r^{D_loc:.2f}")
for k in CAND[1:]:
    c = cdf_of(by("attractor", g2_MHz=101.18) + by("attractor", g2_MHz=101.2), k)
    print(f"   same for the {k}: " + ", ".join(f"{mu_at(c, r):.1e}" for r in r_grid))

# 6. Escape routes and lifetimes above $g_\mathrm{NS}$
For each escaping run, capture = the last iterate farther than 0.3 from the node. We record the distance to the node right after capture, the smallest distance to the $\omega_a/2$ saddle in the 30 iterates before capture, and the number of iterates from capture to landing.
Lifetimes: exponential estimate with censoring, $\tau = (\sum t)/n_\mathrm{esc}$, from these runs (24 per $g_2$) and from `bubble_crisis_lifetime_22_10_2026.json` (80 per $g_2$, censored at $1490/\kappa_b$). Power-law fits $\tau = A(g_2 - g_\ast)^{-\gamma}$, weighted by $\sqrt{n_\mathrm{esc}}$.
**Hole test:** if the escape rate is the attractor's measure inside the node's basin, then $\tau\,\mu_\mathrm{att}(<\rho_{50})/T_a$ is a constant of order 1 (the shape of the basin, which is not a ball, enters only through this constant).

In [ ]:
ESC = {}
for g in TASKS["escape"]["g2_MHz"]:
    i = G.index(g)
    Sn = np.array(ORB["node"][i]["states"] + ORB["node"][i]["partner"]); Ss = np.array(ORB["saddle"][i]["states"] + ORB["saddle"][i]["partner"])
    rows = []
    for x in by("escape", g2_MHz=g):
        if x["landed_Ta"] is None:
            continue
        P = np.array(x["pre"])
        dn = np.linalg.norm(P[:, None] - Sn[None], axis=2).min(1); ds = np.linalg.norm(P[:, None] - Ss[None], axis=2).min(1)
        far = np.where(dn > 0.3)[0]
        if len(far):
            k = far[-1]; rows.append((dn[k + 1], ds[max(0, k - 30):k + 1].min(), len(dn) - k - 1))
    rows = np.array(rows)
    ESC[g] = dict(rows=rows, d_ns=np.linalg.norm(Sn[:, None] - Ss[None], axis=2).min())
    print(f"  {g:7.2f} MHz: node-saddle distance {ESC[g]['d_ns']:.3f} | at capture: distance to the node {np.median(rows[:, 0]):.3f} (median), "
          f"min distance to the saddle in the 30 iterates before {np.median(rows[:, 1]):.3f} (median), {rows[:, 1].min():.3f} (min) | {np.median(rows[:, 2]):.0f} iterates to landing")

TAU = []
for g in TASKS["escape"]["g2_MHz"]:
    t = [x["landed_Ta"] for x in by("escape", g2_MHz=g)]; ne = sum(v is not None for v in t)
    TAU.append((g, sum(v if v is not None else TASKS["escape"]["N_max_Ta"] for v in t)/ne*KT_A, ne, 0))
C22 = json.load(open(OUT_DIR + "torus_crisis/bubble_crisis_lifetime_22_10_2026.json"))
for g in sorted({x["g2_MHz"] for x in C22["runs"]}):
    t = [x["t_esc_Ta"] for x in C22["runs"] if x["g2_MHz"] == g]; ne = sum(v is not None for v in t)
    if ne and g < 103.5:
        TAU.append((g, sum(v if v is not None else C22["N_max_Ta"] for v in t)/ne*KT_A, ne, 1))
TAU = np.array(TAU)
def pl_fit(gc, T=TAU):
    w = np.sqrt(T[:, 2]); p = np.polyfit(np.log(T[:, 0] - gc), np.log(T[:, 1]), 1, w=w)
    return p, np.sqrt(np.sum(w**2*(np.log(T[:, 1]) - np.polyval(p, np.log(T[:, 0] - gc)))**2)/np.sum(w**2))
from scipy.optimize import minimize_scalar
g_free = minimize_scalar(lambda gc: pl_fit(gc)[1], bounds=(100.9, TAU[:, 0].min() - 1e-3), method="bounded").x
FITS = {"free": (g_free, *pl_fit(g_free)), "g_NS": (g_NS, *pl_fit(g_NS))}
T_old = TAU[TAU[:, 0] > 101.45]
FITS["101.4 (22/10)"] = (101.4, *pl_fit(101.4, T_old))
for k, (gc, p, rms) in FITS.items():
    print(f"fit with g_* {k:14s}: g_* = {gc:.3f} MHz, gamma = {-p[0]:.2f}, weighted rms in ln tau = {rms:.2f}" + ("  (only g2 > 101.45 MHz)" if k.startswith("101.4") else ""))
print(f"prediction of the hole picture: gamma = (exponent of rho_50) x (exponent of mu_att) = {p_rho[0]:.2f} x {D_loc:.2f} = {p_rho[0]*D_loc:.2f}")
print(" g2 (MHz) | tau (1/kappa_b) | n_esc | rho_50 | mu_att(<rho_50) | tau mu_att / T_a")
for g, tau, ne, src in TAU[TAU[:, 3] == 0]:
    print(f"  {g:7.2f} | {tau:9.0f} | {ne:2.0f} | {RHO[g]:.4f} | {mu_at(CDF_NS, RHO[g]):.2e} | {tau*mu_at(CDF_NS, RHO[g])/KT_A:6.2f}")

# 7. Figure
(a) Below $g_\mathrm{NS}$: distance to the node after a $10^{-5}$ kick. (b) Above: fraction of kicks of size $\epsilon$ that come back. (c) $\rho_{50}$ against $g_2 - g_\mathrm{NS}$.
(d) Lifetimes with the three fits. (e) Hole test, $\tau\,\mu_\mathrm{att}(<\rho_{50})/T_a$. (f) The section near the node at 101.5 MHz: the attractor (taken at 101.2 MHz), the node, the saddle, and the last 40 iterates before capture of 8 escaping runs.

In [ ]:
amp = lambda Y: (np.asarray(Y)[..., 0] + 1j*np.asarray(Y)[..., 2])/(np.sqrt(2)*phi_a)
fig, axes = plt.subplots(2, 3, figsize=(3.3*figsize_car[0], 2.3*figsize_car[1]))
ax = axes[0, 0]
for g, c in zip(TASKS["ns"]["g2_below"], ("0.6", Perso_palette_float[4], Perso_palette_float[2])):
    for j, x in enumerate(by("ns_below", g2_MHz=g)):
        d = np.array(x["dist_every_10Ta"]); ax.semilogy(10*np.arange(1, len(d) + 1), d, color=c, lw=0.6, label=f"{g} MHz" if j == 0 else None)
ax.set_xscale("log"); ax.set_xlabel(r"$t/T_a$"); ax.set_ylabel("distance to the node"); ax.legend(fontsize=5)
ax.set_title(rf"below $g_\mathrm{{NS}} = {g_NS:.2f}$ MHz: kick of $10^{{-5}}$", fontsize=6)
ax = axes[0, 1]
cm = plt.get_cmap("viridis")
for j, g in enumerate(FR):
    ax.semilogx(EPS, FR[g], "o-", ms=2, lw=0.8, color=cm(j/(len(FR) - 1)), label=f"{g} MHz")
ax.axhline(0.5, color="0.5", lw=0.5, ls=":"); ax.set_xlabel(r"kick $\epsilon$ ($x^\lambda$ units)"); ax.set_ylabel("fraction back on the node"); ax.legend(fontsize=4.5)
ax.set_title(r"above $g_\mathrm{NS}$: basin of the node", fontsize=6)
ax = axes[0, 2]
ax.loglog(g_rho - g_NS, rho, "ko", ms=3)
xx = np.logspace(np.log10(0.01), np.log10(2.5), 50)
ax.loglog(xx, np.exp(np.polyval(p_rho, np.log(xx))), "k--", lw=0.7, label=rf"$\propto (g_2 - g_\mathrm{{NS}})^{{{p_rho[0]:.2f}}}$")
ax.axhline(ESC[101.5]["d_ns"], color="darkorange", lw=0.7, ls=":", label="node–saddle distance")
ax.set_xlabel(r"$(g_2 - g_\mathrm{NS})/2\pi$ (MHz)"); ax.set_ylabel(r"$\rho_{50}$"); ax.legend(fontsize=5)
ax = axes[1, 0]
for src, mk, lab in ((0, "o", "this run (24 per $g_2$)"), (1, "s", "22/10 (80 per $g_2$)")):
    m = TAU[:, 3] == src
    ax.errorbar(TAU[m, 0], TAU[m, 1], yerr=TAU[m, 1]/np.sqrt(TAU[m, 2]), fmt=mk, ms=3, color="k" if src == 0 else "0.55", lw=0.6, label=lab)
xx = np.linspace(101.205, 103.5, 200)
for (k, (gc, p, rms)), ls in zip(FITS.items(), ("-", "--", ":")):
    x_ = xx[xx > gc + 1e-3]
    ax.plot(x_, np.exp(np.polyval(p, np.log(x_ - gc))), ls, color=Perso_palette_float[2], lw=0.8, label=rf"$g_\ast$ {k}: {gc:.2f} MHz, $\gamma = {-p[0]:.2f}$")
ax.axvline(g_NS, color="k", lw=0.5, ls=":")
ax.set_yscale("log"); ax.set_xlabel(r"$g_2/2\pi$ (MHz)"); ax.set_ylabel(r"$\kappa_b\tau$"); ax.legend(fontsize=4.5); ax.set_xlim(101.1, 103.2)
ax = axes[1, 1]
m = TAU[:, 3] == 0
prod = np.array([tau*mu_at(CDF_NS, RHO[g])/KT_A for g, tau in TAU[m, :2]])
ax.errorbar(TAU[m, 0], prod, yerr=prod/np.sqrt(TAU[m, 2]), fmt="ko", ms=3, lw=0.6, label=r"$\tau\,\mu_\mathrm{att}(<\rho_{50})/T_a$")
ax.set_yscale("log"); ax.set_ylim(0.1, 100); ax.set_xlabel(r"$g_2/2\pi$ (MHz)"); ax.legend(fontsize=5)
ax.set_title(rf"hole test ($\tau$ itself varies by $\times{TAU[m, 1].max()/TAU[m, 1].min():.0f}$)", fontsize=6)
ax = axes[1, 2]
i15 = G.index(101.5)
Sn = np.array(ORB["node"][i15]["states"] + ORB["node"][i15]["partner"]); Ss = np.array(ORB["saddle"][i15]["states"] + ORB["saddle"][i15]["partner"])
a_n = amp(Sn[0])
att = np.concatenate([amp(np.array(x["points"])) for x in by("attractor", g2_MHz=101.2)])
ax.scatter(att.real, att.imag, s=1.0, color="0.65", lw=0, label="attractor at 101.2 MHz")
for x in [x for x in by("escape", g2_MHz=101.5) if x["landed_Ta"] is not None][:12]:
    P = np.array(x["pre"]); dn = np.linalg.norm(P[:, None] - Sn[None], axis=2).min(1)
    k = np.where(dn > 0.3)[0][-1]; n_ = np.arange(max(0, k - 40), min(len(P), k + 60)); tr = amp(P[n_])
    sc = ax.scatter(tr.real, tr.imag, c=n_ - k, cmap="coolwarm", vmin=-40, vmax=60, s=3, lw=0)
plt.colorbar(sc, ax=ax, label="iterate $-$ capture", shrink=0.8)
ax.plot(*np.array([[z.real, z.imag] for z in amp(Sn)]).T, "o", color=Perso_palette_float[3], ms=4, label=r"$\omega_a/2$ node (101.5 MHz)")
ax.plot(*np.array([[z.real, z.imag] for z in amp(Ss)]).T, "^", color="darkorange", ms=4, label=r"$\omega_a/2$ saddle")
ax.set_xlim(a_n.real - 0.6, a_n.real + 0.6); ax.set_ylim(a_n.imag - 0.6, a_n.imag + 0.6); ax.set_aspect("equal")
ax.set_xlabel(r"Re $a(2\pi k)$"); ax.set_ylabel(r"Im $a(2\pi k)$"); ax.legend(fontsize=4.5, loc="lower left", markerscale=1)
ax.set_title("101.5 MHz: 12 escapes around their capture", fontsize=6)
plt.tight_layout()
plt.savefig(FIG_DIR + "crisis_collision_eps_d10_23_10_2026.pdf", bbox_inches="tight", dpi=300)
plt.show()

In [ ]:
json.dump({"description": "Crisis of the bubble's chaotic attractor at eps_d = 10 g2 (kappa_b = pi/10.4 code). From crisis_collision.ipynb; raw runs in crisis_collision_23_10_2026.json",
           "g_NS_MHz": g_NS, "g_NS1_MHz": g_NS1, "g_fold_MHz": B2["g2_omega_half_fold_MHz"],
           "rho50": {str(g): r for g, r in RHO.items()}, "rho50_exponent": p_rho[0], "mu_att_exponent": D_loc,
           "tau": [dict(g2_MHz=g, tau_over_kappa=t, n_esc=int(ne), source=["23_10", "22_10"][int(s)]) for g, t, ne, s in TAU],
           "fits": {k: dict(g_star=gc, gamma=-p[0], rms=rms) for k, (gc, p, rms) in FITS.items()}},
          open(OUT_DIR + "torus_crisis/crisis_collision_summary_23_10_2026.json", "w"))

## Conclusions
**1. There is no collision with a saddle.** Scenario (A) fails on all three tests.
- The $\omega_a/2$ saddle sits only 0.08–0.10 from the node (they come from the same saddle-node).
- Above $g_\mathrm{NS}$, **neither** branch of its unstable manifold goes to the node directly. Both enter the chaotic set and land only after $10^3$–$10^5\,T_a$.
  Only in the first stability window (99.97–100.24 MHz), just after the saddle-node, does one branch go straight to the node (90–150 $T_a$).
- Escaping trajectories do not pass near the saddle before capture: in the last 30 iterates they stay 0.13–0.23 from it (median), farther than the node itself is.
  They jump from the chaotic set straight to $\simeq 0.1$ from the node, then spiral in over 110–200 iterates.
- Below $g_\mathrm{NS}$ the attractor comes within 0.004–0.024 of **every** candidate (node, saddle, branch I, satellite) in $8\times10^5$ iterates, with comparable weights $\mu_\mathrm{att}(<0.05) \sim 10^{-4}$. They are all embedded in the chaotic set, and none of them stands out as a mediating orbit.

**2. What happens: the node becomes stable inside the attractor.** For $g_2 < g_\mathrm{NS} = 101.2014$ MHz the $\omega_a/2$ node is unstable (Neimark–Sacker) and is visited by the attractor like any embedded orbit.
When it restabilises, its basin is a hole in the attractor, and the chaos becomes transient from there on:
- $\tau = 1.1\times10^5$, $4.9\times10^4$ and $1.8\times10^4/\kappa_b$ at 101.22, 101.25 and 101.3 MHz.
- The $g_c \simeq 101.4$ MHz of 22/10 was an artefact of the cutoff at $1490/\kappa_b$, which cannot see lifetimes of $10^4$–$10^5/\kappa_b$. With all the data, a threshold at 101.4 MHz is excluded, since there are escapes at 101.22–101.35 MHz.

**3. The Neimark–Sacker point and the hole.**
- Just below $g_\mathrm{NS}$ (101.18 MHz) a $10^{-5}$ kick settles on a small invariant set around the node, of size 0.024 (the Neimark–Sacker is supercritical). At 101.15 MHz it has gone, and the kick leaves for the chaotic set. The small torus survives only 0.02–0.05 MHz below $g_\mathrm{NS}$.
- Above $g_\mathrm{NS}$ the node's basin, measured by random kicks, has a half-return radius $\rho_{50}$ that grows from 0.007 at +0.02 MHz to 0.1 at +1.8 MHz, as $(g_2 - g_\mathrm{NS})^{0.60}$. It stays below the node–saddle distance up to about 103 MHz.
- The attractor's weight near the node scales as $\mu_\mathrm{att}(<r) \propto r^{2.75}$ (its local dimension around the node).
- If the escape rate is the attractor's weight inside the hole, then $\tau \propto (g_2 - g_\mathrm{NS})^{-\gamma}$ with $\gamma = 0.60\times2.75 = 1.64$.
  The lifetimes give $\gamma = 1.60$ with $g_\ast = g_\mathrm{NS}$ fixed (rms 0.22 in $\ln\tau$), and $g_\ast = 101.15$ MHz, $\gamma = 1.85$ with $g_\ast$ free (rms 0.17).
  Over 101.25–103 MHz, $\tau$ changes by a factor of 530 while $\tau\,\mu_\mathrm{att}(<\rho_{50})/T_a$ stays at 3–9, i.e. about one capture per few entries into a ball of radius $\rho_{50}$.
  The two outliers are explained by the data: at 101.22 MHz there are only 3 escapes and $\mu_\mathrm{att}$ is extrapolated below the sampled $r$; at 101.4 MHz $\rho_{50}$ falls on a coarse step of the $\epsilon$ grid.
- The threshold is therefore between 101.15 and 101.20 MHz. Whether the small torus just below $g_\mathrm{NS}$ already opens the hole is not resolved: a capture onto it is not counted as a landing, and none shows up in the distance histograms at 101.18–101.20 MHz.

**4. The same mechanism in the first stability window of the node (99.97–100.24 MHz).** The node is born at the saddle-node inside the region the attractor visits, and the attractor runs land on it:
1/8, 2/8 and 6/8 runs within $10^5\,T_a$ at 100.0, 100.1 and 100.2 MHz, i.e. $\tau \sim 10^4$–$10^5/\kappa_b$. At 100.3 MHz 6 of 8 runs end close to the node (probably on the torus born at the Neimark–Sacker at 100.24 MHz; not checked).
So, on time scales of $10^5\,T_a$, the bubble's chaotic set is an attractor at $\varepsilon_d = 10\,g_2$ only for $g_2 < 99.97$ MHz and on about 100.4–101.15 MHz.

**5. Summary.** The end of the bubble's chaotic attractor is not a boundary crisis with a saddle. A periodic orbit embedded in the attractor (the $\omega_a/2$ orbit) becomes stable, and its basin, which grows from almost nothing, opens a hole in the chaotic set.
The lifetime exponent is fixed by how fast the basin grows ($\rho_{50} \propto \delta g^{0.6}$) and by the local dimension of the attractor around the orbit (2.75).